<a href="https://colab.research.google.com/github/Samarth-27/Celebal-CEI/blob/main/week7_SamarthJain_Jecrc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — Retrieval-Augmented Generation (RAG) Document Question Answering


In [1]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00


In [2]:
import fitz, numpy as np, faiss, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 1. Document Ingestion

In [3]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

text = ""
with fitz.open(pdf_path) as pdf:
    for page in pdf:
        text += page.get_text()
print(f"Extracted {len(text)} characters")

Saving SamarthJain resume.pdf to SamarthJain resume.pdf
Extracted 3618 characters


## 2. Text Chunking

In [4]:
def chunk_text(text, size=500, overlap=50):
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size])
        i += size - overlap
    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))

Total chunks: 9


## 3. Embedding Creation

In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, show_progress_bar=True).astype("float32")
faiss.normalize_L2(embeddings)
print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (9, 384)


## 4. Vector Database

In [6]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS index ready —", index.ntotal, "vectors")

FAISS index ready — 9 vectors


## 5. Query Processing

In [7]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve(question, k=10, top=3):
    q_emb = embedder.encode([question]).astype("float32")
    faiss.normalize_L2(q_emb)
    _, idx = index.search(q_emb, min(k, index.ntotal))
    candidates = [chunks[i] for i in idx[0]]
    scores = reranker.predict([(question, c) for c in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:top]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 6. Context Retrieval

In [8]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
print("Model loaded!")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded!


## 7. Answer Generation

In [9]:
def ask(question):
    context = "\n\n".join(retrieve(question))
    messages = [{"role": "user", "content": f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {question}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("Q:", question)
    print("A:", response.strip())

ask("What projects has the candidate built?")
ask("What is the candidate's tech stack?")
ask("What hackathon did the candidate participate in?")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: What projects has the candidate built?
A: The candidate has built the following projects:

1. StayTrack – An AI-Powered Hostel Management System using React, Node.js, Express.js, MongoDB, TypeScript, Tailwind CSS, JWT, Razorpay, and AI (LLM + RAG). They built a role-based hostel management system with OpenAI API, LangChain, and LlamaIndex, and mastered prompt engineering techniques. They won the prize in JECRC HackQuest 8.0.

2. Unspecified projects in data preprocessing, model building, evaluation workflows, and project-based learning in classification and clustering.

3. Participated in Smart India Hackathon 2025 as a runner-up, where they built and presented a problem-solving project as part of a team.

4. Currently interning at Celebal Technologies as a CEI
Q: What is the candidate's tech stack?
A: The candidate has experience with the following technologies based on the context provided:

* React
* JavaScript
* Node.js
* Express.js
* MongoDB
* TypeScript
* Tailwind CSS
* JWT
* 

## Conclusion
This project demonstrates a complete end-to-end RAG pipeline from PDF ingestion to answer generation using Mistral-7B. The system accurately answered questions about a resume without hallucination, which highlights the core advantage of RAG over plain LLMs.